In [1]:
!nvidia-smi

Sun Jul  5 11:48:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!unzip dataset.zip

Archive:  dataset.zip
  inflating: IMDB Dataset.csv        


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv('IMDB Dataset.csv')

In [5]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [7]:
df.isna().sum()

,0
review,0
sentiment,0


In [8]:
df['sentiment'].unique()

array(['positive', 'negative'], dtype=object)

In [9]:
df['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [10]:
df.drop_duplicates(inplace=True)
df = df.reset_index(drop=True)

# Text Cleaning

* convert string into lower case

In [11]:
df['review'] = df['review'].str.lower()

* Remove HTML elements

In [12]:
from bs4 import BeautifulSoup

def remove_html(text):
  if isinstance(text, str):
      return BeautifulSoup(text, "html.parser").get_text(separator=" ")
  return text

In [13]:
df['review'] = df['review'].apply(remove_html)

* remove urls

In [14]:
import re

def remove_url(text):
  return re.sub(r'https?://\S+|www\.\S+', '', text)

In [15]:
df['review'] = df['review'].apply(remove_url)

* remove punctuation

In [16]:
import string

def remove_punch(text):
  return text.translate(str.maketrans('', '', string.punctuation))

In [17]:
df['review'] = df['review'].apply(remove_punch)

* remove numbers

In [18]:
import re

def remove_num(text):
  return re.sub(r'\d+', '', text)

In [19]:
df['review'] = df['review'].apply(remove_num)

* remove emojis

In [20]:
import re

def remove_emoji(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # Emoticons
        u"\U0001F300-\U0001F5FF"  # Symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # Transport & map
        u"\U0001F1E0-\U0001F1FF"  # Flags
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE,
    )
    return emoji_pattern.sub("", text)

In [21]:
df['review'] = df['review'].apply(remove_emoji)

* remove extra space

In [22]:
import re

def remove_extra_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

In [23]:
df["review"] = df["review"].apply(remove_extra_spaces)

* tokenization

In [24]:
import nltk
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [25]:
from nltk.tokenize import word_tokenize

df["review"] = df["review"].apply(word_tokenize)

In [26]:
df.head()

,review,sentiment
0,"[one, of, the, other, reviewers, has, mentione...",positive
1,"[a, wonderful, little, production, the, filmin...",positive
2,"[i, thought, this, was, a, wonderful, way, to,...",positive
3,"[basically, theres, a, family, where, a, littl...",negative
4,"[petter, matteis, love, in, the, time, of, mon...",positive


* Convert Text to Numbers

In [27]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [28]:
tokenizer = Tokenizer(
    num_words=10000,
    oov_token="<OOV>"
)

In [29]:
tokenizer.fit_on_texts(df["review"])

In [30]:
sequences = tokenizer.texts_to_sequences(df["review"])

In [31]:
df["sequence"] = sequences

In [32]:
len(tokenizer.word_index)

162913

In [33]:
for word, index in list(tokenizer.word_index.items())[:20]:
    print(word, ":", index)

<OOV> : 1
the : 2
and : 3
a : 4
of : 5
to : 6
is : 7
in : 8
it : 9
i : 10
this : 11
that : 12
was : 13
as : 14
with : 15
for : 16
movie : 17
but : 18
film : 19
on : 20


In [34]:
df.head()

,review,sentiment,sequence
0,"[one, of, the, other, reviewers, has, mentione...",positive,"[28, 5, 2, 76, 1956, 45, 1055, 12, 101, 145, 4..."
1,"[a, wonderful, little, production, the, filmin...",positive,"[4, 383, 117, 356, 2, 1322, 2888, 7, 52, 1, 52..."
2,"[i, thought, this, was, a, wonderful, way, to,...",positive,"[10, 194, 11, 13, 4, 383, 95, 6, 1118, 59, 20,..."
3,"[basically, theres, a, family, where, a, littl...",negative,"[665, 217, 4, 235, 115, 4, 117, 434, 3519, 119..."
4,"[petter, matteis, love, in, the, time, of, mon...",positive,"[1, 1, 110, 8, 2, 59, 5, 288, 7, 4, 2066, 1362..."


In [35]:
df["review"].apply(len).max()

2450

In [36]:
review_lengths = df["review"].apply(len)

print(review_lengths.describe())

count    49582.000000
mean       227.179642
std        168.526062
min          4.000000
25%        124.000000
50%        170.000000
75%        275.750000
max       2450.000000
Name: review, dtype: float64


In [37]:
max_len= 1000

In [38]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 300   # Replace with the value we decide

X = pad_sequences(
    df["sequence"],
    maxlen=max_len,
    padding="post",
    truncating="post"
)

print(X.shape)

(49582, 300)


In [39]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(df["sentiment"])

print(y[:10])

[1 1 1 0 1 1 1 0 0 1]


In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(39665, 300)
(9917, 300)


In [41]:
vocab_size = len(tokenizer.word_index) + 1
print(vocab_size)

162914


In [42]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

model = Sequential([
    Input(shape=(max_len,)),
    Embedding(input_dim=vocab_size, output_dim=128),
    LSTM(64),
    Dense(1, activation="sigmoid")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 300, 128)       │    20,852,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,902,465 (79.74 MB)

 Trainable params: 20,902,465 (79.74 MB)

 Non-trainable params: 0 (0.00 B)

In [43]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [44]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 37s 26ms/step - accuracy: 0.5184 - loss: 0.6916 - val_accuracy: 0.5514 - val_loss: 0.6896
Epoch 2/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 33s 27ms/step - accuracy: 0.5408 - loss: 0.6815 - val_accuracy: 0.5356 - val_loss: 0.6890
Epoch 3/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 33s 27ms/step - accuracy: 0.8044 - loss: 0.4321 - val_accuracy: 0.8698 - val_loss: 0.3045
Epoch 4/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 41s 27ms/step - accuracy: 0.9102 - loss: 0.2333 - val_accuracy: 0.8884 - val_loss: 0.2783
Epoch 5/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 41s 26ms/step - accuracy: 0.9373 - loss: 0.1713 - val_accuracy: 0.8865 - val_loss: 0.3037
Epoch 6/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 33s 27ms/step - accuracy: 0.9587 - loss: 0.1211 - val_accuracy: 0.8771 - val_loss: 0.3510
Epoch 7/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 33s 26ms/step - accuracy: 0.9740 - loss: 0.0834 - val_accuracy: 0.8750 - val_loss: 0.3863
Epoch 8/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 33s 27ms/step - accuracy: 0.9838 -

In [45]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

310/310 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.8718 - loss: 0.5507
Test Loss: 0.5507170557975769
Test Accuracy: 0.871836245059967


In [46]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_sentiment(text):
    # Convert text into sequence
    sequence = tokenizer.texts_to_sequences([text])

    # Pad sequence
    padded = pad_sequences(sequence, maxlen=max_len, padding='post', truncating='post')

    # Predict
    prediction = model.predict(padded, verbose=0)[0][0]

    if prediction >= 0.5:
        print(f"Positive 😊 ({prediction:.4f})")
    else:
        print(f"Negative 😞 ({prediction:.4f})")

In [47]:
predict_sentiment("This movie was absolutely amazing. I loved it.")

Positive 😊 (0.9950)


In [52]:
predict_sentiment("Worst movie ever.")

Negative 😞 (0.0023)


In [49]:
model.save('review.keras')